In [1]:
# ============================================================
# CELL 1 — Imports & SparkSession
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, hour, dayofweek, to_timestamp, count, sum as spark_sum,
    mean, stddev, abs as spark_abs, lag, unix_timestamp,
    when, lit, percentile_approx
)
from pyspark.sql.window import Window
import warnings
warnings.filterwarnings("ignore")

spark = SparkSession.builder \
    .appName("FRAUD-X42") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession OK")

SparkSession OK


In [2]:
# ============================================================
# CELL 2 — Étape 1 : Chargement & Exploration du dataset
# ============================================================

# Chargement du CSV
df = spark.read.csv("nexbuy_transactions.csv", header=True, inferSchema=True)

# Nombre de lignes et colonnes
print(f"Lignes : {df.count()} | Colonnes : {len(df.columns)}")

# Types inférés par Spark
df.printSchema()

# Statistiques descriptives des colonnes numériques
df.describe("amount_eur", "nb_items", "account_age_days").show()

# Valeurs manquantes par colonne
from pyspark.sql.functions import isnan, isnull
print("=== Valeurs manquantes ===")
for c in df.columns:
    nulls = df.filter(isnull(col(c))).count()
    if nulls > 0:
        print(f"  {c} : {nulls} nulls")

# Plage de dates
df.selectExpr("min(timestamp)", "max(timestamp)").show()

# Montant min / max / médian
df.selectExpr("min(amount_eur)", "max(amount_eur)", 
              "percentile_approx(amount_eur, 0.5) as median_amount").show()

Lignes : 4970 | Colonnes : 16
root
 |-- transaction_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- amount_eur: double (nullable = true)
 |-- product_category: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- nb_items: integer (nullable = true)
 |-- is_new_account: boolean (nullable = true)
 |-- account_age_days: integer (nullable = true)
 |-- delivery_country: string (nullable = true)
 |-- billing_country: string (nullable = true)
 |-- is_fraud: boolean (nullable = true)

+-------+------------------+-----------------+-----------------+
|summary|        amount_eur|         nb_items| account_age_days|
+-------+------------------+-----------------+-----------------+
|  count|              4970|             4970|             4

In [3]:
# ============================================================
# CELL 3 — Étape 2 : Prétraitement & Nettoyage
# ============================================================

# 1. Imputer les nulls sur les colonnes catégorielles
df_clean = df \
    .fillna("inconnu", subset=["product_category", "device_type"]) \
    .fillna("ip_masquee", subset=["ip_address"])

# 2. Filtrer les montants aberrants
#    - On garde : 0 < amount_eur < 10 000
#    - Justification : médiane = 46€, max légitime estimé à 10 000€
#      les valeurs négatives et > 10 000 sont des anomalies de saisie
SEUIL_MAX = 10000
df_clean = df_clean.filter((col("amount_eur") > 0) & (col("amount_eur") < SEUIL_MAX))

# 3. Extraire heure et jour de la semaine depuis timestamp
df_clean = df_clean \
    .withColumn("hour", hour(col("timestamp"))) \
    .withColumn("day_of_week", dayofweek(col("timestamp")))

# 4. Vérification post-nettoyage
print(f"Lignes après nettoyage : {df_clean.count()}")
df_clean.describe("amount_eur").show()

# 5. Mise en cache — on réutilise ce DataFrame dans toutes les pistes
#    Evite de relire et retraiter le CSV à chaque action
df_clean.cache()
print("DataFrame mis en cache.")

Lignes après nettoyage : 4962
+-------+------------------+
|summary|        amount_eur|
+-------+------------------+
|  count|              4962|
|   mean| 90.76628778718269|
| stddev|195.97685857488355|
|    min|               5.0|
|    max|           2498.73|
+-------+------------------+

DataFrame mis en cache.


In [4]:
# ============================================================
# CELL 4 — Piste A : Profil Horaire (Z-score)
# ============================================================
from pyspark.sql.functions import avg, pow as spark_pow, sqrt

# Compter les transactions par heure
hourly = df_clean.groupBy("hour").count().orderBy("hour")

# Calculer moyenne et écart-type du volume horaire
stats = hourly.selectExpr("avg(count) as mu", "stddev(count) as sigma").collect()[0]
mu, sigma = stats["mu"], stats["sigma"]
print(f"Moyenne horaire : {mu:.2f} | Stddev : {sigma:.2f}")

# Calculer le z-score par heure
hourly_z = hourly.withColumn(
    "z_score", (col("count") - lit(mu)) / lit(sigma)
)

# Heures aberrantes : z > 2
print("\n=== Heures suspectes (z > 2) ===")
hourly_z.filter(col("z_score") > 2).orderBy(col("z_score").desc()).show()

# Afficher toutes les heures pour visualisation
print("\n=== Volume par heure (toutes heures) ===")
hourly_z.orderBy("hour").show(24)

Moyenne horaire : 206.75 | Stddev : 14.27

=== Heures suspectes (z > 2) ===
+----+-----+-------+
|hour|count|z_score|
+----+-----+-------+
+----+-----+-------+


=== Volume par heure (toutes heures) ===
+----+-----+--------------------+
|hour|count|             z_score|
+----+-----+--------------------+
|   0|  204| -0.1926514584545923|
|   1|  213|  0.4378442237604371|
|   2|  230|   1.628780512388826|
|   3|  231|  1.6988355881904957|
|   4|  195| -0.8231471406696217|
|   5|  227|   1.418615284983816|
|   6|  198| -0.6129819132646119|
|   7|  209| 0.15762392055375735|
|   8|  186| -1.4536428228846512|
|   9|  221|  0.9982848301737965|
|  10|  210| 0.22767899635542727|
|  11|  233|  1.8389457397938358|
|  12|  213|  0.4378442237604371|
|  13|  204| -0.1926514584545923|
|  14|  193| -0.9632572922729615|
|  15|  209| 0.15762392055375735|
|  16|  204| -0.1926514584545923|
|  17|  187| -1.3835877470829812|
|  18|  195| -0.8231471406696217|
|  19|  204| -0.1926514584545923|
|  20|  186| -1

In [5]:
# ============================================================
# CELL 5 — Piste B : Comportement Client (Burst)
# ============================================================

# Fenêtre glissante par user_id, triée par timestamp
w = Window.partitionBy("user_id").orderBy(unix_timestamp("timestamp"))

# Timestamp de la transaction précédente du même user
df_b = df_clean.withColumn("prev_ts", lag(unix_timestamp("timestamp"), 1).over(w))

# Durée en secondes depuis la transaction précédente
df_b = df_b.withColumn("delta_sec", unix_timestamp("timestamp") - col("prev_ts"))

# Flag burst : < 2h (7200 secondes) entre deux transactions
df_b = df_b.withColumn("is_burst", col("delta_sec") < 7200)

# Compter les bursts par user
burst_users = df_b.filter(col("is_burst") == True) \
    .groupBy("user_id") \
    .count() \
    .filter(col("count") > 5) \
    .orderBy(col("count").desc())

print("=== Users avec > 5 transactions en moins de 2h ===")
burst_users.show(20)

# Cumul > 1500€ en 24h par user
daily_amount = df_clean \
    .withColumn("date", col("timestamp").cast("date")) \
    .groupBy("user_id", "date") \
    .agg(spark_sum("amount_eur").alias("daily_total")) \
    .filter(col("daily_total") > 1500) \
    .orderBy(col("daily_total").desc())

print("=== Users avec cumul > 1500€ en 24h ===")
daily_amount.show(20)

=== Users avec > 5 transactions en moins de 2h ===
+--------+-----+
| user_id|count|
+--------+-----+
|USR-0175|   11|
|USR-0194|   10|
|USR-0063|    9|
|USR-0078|    8|
|USR-0415|    8|
|USR-0301|    7|
|USR-0523|    7|
|USR-0432|    6|
+--------+-----+

=== Users avec cumul > 1500€ en 24h ===
+--------+----------+------------------+
| user_id|      date|       daily_total|
+--------+----------+------------------+
|USR-0175|2026-03-14| 4993.450000000001|
|USR-0194|2026-05-08|           4963.41|
|USR-0063|2026-05-22|           3963.75|
|USR-0415|2026-04-12|3858.0800000000004|
|USR-0087|2026-05-14|           3638.95|
|USR-0078|2026-05-12|           3481.61|
|USR-0432|2026-04-15|           3162.95|
|USR-0523|2026-04-15|           2879.13|
|USR-0301|2026-04-17|2805.0600000000004|
|USR-0112|2026-04-09|           2498.73|
|USR-0267|2026-03-14|           2498.62|
|USR-0421|2026-03-27|           2370.36|
|USR-0389|2026-04-10|           2359.56|
|USR-0421|2026-04-12|            2353.3|
|USR-01

In [6]:
# ============================================================
# CELL 6 — Piste C : Géographie Suspecte
# ============================================================

# Pays à risque mentionnés dans le brief
high_risk_countries = ["RO", "UA", "MD", "BY", "XK"]

# Condition 1 : billing != delivery
# Condition 2 : mode de paiement suspect
suspicious_payments = ["carte_prepayee", "crypto"]

df_c = df_clean.withColumn(
    "geo_mismatch", col("billing_country") != col("delivery_country")
).withColumn(
    "suspicious_payment", col("payment_method").isin(suspicious_payments)
).withColumn(
    "high_risk_delivery", col("delivery_country").isin(high_risk_countries)
)

# Transactions suspectes : mismatch ET paiement suspect
geo_suspect = df_c.filter(
    col("geo_mismatch") & col("suspicious_payment")
)

print(f"Transactions geo-suspectes (mismatch + paiement suspect) : {geo_suspect.count()}")

# Croisement avec pays à risque
geo_high_risk = df_c.filter(
    col("geo_mismatch") & col("suspicious_payment") & col("high_risk_delivery")
)

print(f"Dont livraison vers pays à risque : {geo_high_risk.count()}")

# Distribution par pays de livraison
print("\n=== Répartition par pays de livraison (suspects) ===")
geo_suspect.groupBy("delivery_country") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(20)

# Afficher les transactions les plus suspectes
print("\n=== Échantillon transactions géo-suspectes ===")
geo_suspect.select(
    "transaction_id", "user_id", "amount_eur",
    "billing_country", "delivery_country", "payment_method"
).orderBy(col("amount_eur").desc()).show(10)

Transactions geo-suspectes (mismatch + paiement suspect) : 170
Dont livraison vers pays à risque : 170

=== Répartition par pays de livraison (suspects) ===
+----------------+-----+
|delivery_country|count|
+----------------+-----+
|              XK|   40|
|              UA|   38|
|              RO|   37|
|              BY|   30|
|              MD|   25|
+----------------+-----+


=== Échantillon transactions géo-suspectes ===
+--------------+--------+----------+---------------+----------------+--------------+
|transaction_id| user_id|amount_eur|billing_country|delivery_country|payment_method|
+--------------+--------+----------+---------------+----------------+--------------+
|     TXN-01271|USR-0112|   2498.73|             DE|              XK|        crypto|
|     TXN-02812|USR-0267|   2498.62|             ES|              BY|carte_prepayee|
|     TXN-03152|USR-0421|   2370.36|             DE|              BY|        crypto|
|     TXN-03976|USR-0389|   2359.56|             FR|       

In [7]:
# ============================================================
# CELL 7 — Piste D : Nouveaux Comptes
# ============================================================

# Séparer nouveaux comptes (< 7 jours) et anciens
df_new = df_clean.filter(col("account_age_days") < 7)
df_old = df_clean.filter(col("account_age_days") >= 7)

print(f"Nouveaux comptes : {df_new.count()} transactions")
print(f"Anciens comptes  : {df_old.count()} transactions")

# Statistiques comparées sur amount_eur
print("\n=== Distribution montants — Nouveaux comptes ===")
df_new.selectExpr(
    "mean(amount_eur) as mean",
    "stddev(amount_eur) as stddev",
    "percentile_approx(amount_eur, 0.5) as median",
    "min(amount_eur) as min",
    "max(amount_eur) as max"
).show()

print("=== Distribution montants — Anciens comptes ===")
df_old.selectExpr(
    "mean(amount_eur) as mean",
    "stddev(amount_eur) as stddev",
    "percentile_approx(amount_eur, 0.5) as median",
    "min(amount_eur) as min",
    "max(amount_eur) as max"
).show()

# Modes de paiement des nouveaux comptes
print("=== Modes de paiement — Nouveaux comptes ===")
df_new.groupBy("payment_method").count() \
    .orderBy(col("count").desc()).show()

# Pays de livraison des nouveaux comptes
print("=== Pays livraison suspects — Nouveaux comptes ===")
high_risk_countries = ["RO", "UA", "MD", "BY", "XK"]
df_new.filter(col("delivery_country").isin(high_risk_countries)) \
    .groupBy("delivery_country").count() \
    .orderBy(col("count").desc()).show()

Nouveaux comptes : 142 transactions
Anciens comptes  : 4820 transactions

=== Distribution montants — Nouveaux comptes ===
+----------------+-----------------+------+------+-------+
|            mean|           stddev|median|   min|    max|
+----------------+-----------------+------+------+-------+
|905.258521126761|664.2024181565673|565.68|152.31|2498.73|
+----------------+-----------------+------+------+-------+

=== Distribution montants — Anciens comptes ===
+-----------------+-----------------+------+---+------+
|             mean|           stddev|median|min|   max|
+-----------------+-----------------+------+---+------+
|66.77087344398342|80.66286515869385| 44.48|5.0|1758.6|
+-----------------+-----------------+------+---+------+

=== Modes de paiement — Nouveaux comptes ===
+--------------+-----+
|payment_method|count|
+--------------+-----+
|carte_prepayee|  108|
|        crypto|   34|
+--------------+-----+

=== Pays livraison suspects — Nouveaux comptes ===
+----------------

In [8]:
# ============================================================
# CELL 8 — Piste E : Score de Risque Composite (0-100)
# ============================================================

high_risk_countries = ["RO", "UA", "MD", "BY", "XK"]
suspicious_payments = ["carte_prepayee", "crypto"]

# Utilisateurs burst (Piste B)
burst_user_ids = [r["user_id"] for r in burst_users.select("user_id").collect()]

# Construire les signaux sur chaque transaction
df_scored = df_clean \
    .withColumn("signal_geo",
        when(
            (col("billing_country") != col("delivery_country")) &
            col("delivery_country").isin(high_risk_countries) &
            col("payment_method").isin(suspicious_payments),
            lit(40)
        ).otherwise(lit(0))
    ) \
    .withColumn("signal_new_account",
        when(col("account_age_days") < 7, lit(30)).otherwise(lit(0))
    ) \
    .withColumn("signal_burst",
        when(col("user_id").isin(burst_user_ids), lit(20)).otherwise(lit(0))
    ) \
    .withColumn("signal_amount",
        when(col("amount_eur") > 500, lit(10)).otherwise(lit(0))
    ) \
    .withColumn("risk_score",
        col("signal_geo") + col("signal_new_account") +
        col("signal_burst") + col("signal_amount")
    )

# Top 50 transactions les plus suspectes
print("=== TOP 50 transactions suspectes ===")
top50 = df_scored.orderBy(col("risk_score").desc()).limit(50)
top50.select(
    "transaction_id", "user_id", "amount_eur", "risk_score",
    "signal_geo", "signal_new_account", "signal_burst", "signal_amount",
    "payment_method", "delivery_country"
).show(50, truncate=False)

# Sauvegarder les transaction_ids pour validation finale
top50_ids = [r["transaction_id"] for r in top50.select("transaction_id").collect()]
print(f"\nNombre de transactions dans le top 50 : {len(top50_ids)}")

=== TOP 50 transactions suspectes ===
+--------------+--------+----------+----------+----------+------------------+------------+-------------+--------------+----------------+
|transaction_id|user_id |amount_eur|risk_score|signal_geo|signal_new_account|signal_burst|signal_amount|payment_method|delivery_country|
+--------------+--------+----------+----------+----------+------------------+------------+-------------+--------------+----------------+
|TXN-00397     |USR-0194|574.06    |100       |40        |30                |20          |10           |carte_prepayee|MD              |
|TXN-00449     |USR-0415|507.96    |100       |40        |30                |20          |10           |carte_prepayee|RO              |
|TXN-00584     |USR-0063|536.02    |100       |40        |30                |20          |10           |carte_prepayee|XK              |
|TXN-01020     |USR-0175|593.12    |100       |40        |30                |20          |10           |carte_prepayee|XK              |
|TX

In [10]:
# ============================================================
# CELL 9 — Étape 5 : Optimisation (cache + shuffle)
# ============================================================
import time

# SANS cache — recharger depuis le CSV brut
df_nocache = spark.read.csv("nexbuy_transactions.csv", header=True, inferSchema=True) \
    .filter((col("amount_eur") > 0) & (col("amount_eur") < 10000)) \
    .withColumn("hour", hour(col("timestamp")))

start = time.time()
df_nocache.groupBy("hour").count().orderBy("hour").collect()
df_nocache.select("transaction_id", "amount_eur").describe().collect()
t_nocache = time.time() - start
print(f"Sans cache : {t_nocache:.2f} secondes")

# AVEC cache — df_clean est déjà en cache depuis Cell 3
start = time.time()
df_clean.groupBy("hour").count().orderBy("hour").collect()
df_clean.select("transaction_id", "amount_eur").describe().collect()
t_cache = time.time() - start
print(f"Avec cache : {t_cache:.2f} secondes")

print(f"\nGain : {((t_nocache - t_cache) / t_nocache * 100):.1f}% plus rapide avec cache")

print("""
=== Justifications optimisation ===
1. CACHE : df_clean.cache() évite de relire et retraiter le CSV
   à chaque action Spark. Bénéfique car df_clean est utilisé
   dans les 5 pistes (actions multiples sur le même DataFrame).

2. PARTITIONNEMENT : spark.sql.shuffle.partitions=8 (Cell 1)
   adapté à un dataset de ~5000 lignes. La valeur par défaut
   (200) créerait 200 partitions vides — overhead inutile.

3. SHUFFLE ÉVITÉ : Piste C utilise .isin() au lieu d'un join
   avec une table pays à risque — évite un shuffle complet
   pour seulement 5 valeurs de référence.
""")

Sans cache : 0.54 secondes
Avec cache : 0.43 secondes

Gain : 20.1% plus rapide avec cache

=== Justifications optimisation ===
1. CACHE : df_clean.cache() évite de relire et retraiter le CSV
   à chaque action Spark. Bénéfique car df_clean est utilisé
   dans les 5 pistes (actions multiples sur le même DataFrame).

2. PARTITIONNEMENT : spark.sql.shuffle.partitions=8 (Cell 1)
   adapté à un dataset de ~5000 lignes. La valeur par défaut
   (200) créerait 200 partitions vides — overhead inutile.

3. SHUFFLE ÉVITÉ : Piste C utilise .isin() au lieu d'un join
   avec une table pays à risque — évite un shuffle complet
   pour seulement 5 valeurs de référence.



In [17]:
# ============================================================
# CELL 10 — Étape 6 : Visualisations
# ============================================================
import matplotlib.pyplot as plt
import pandas as pd

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("FRAUD-X42 — Analyse Forensique NexBuy", fontsize=16, fontweight='bold')

# --- Piste A : Volume horaire ---
hourly_pd = hourly_z.orderBy("hour").toPandas()
axes[0,0].bar(hourly_pd["hour"], hourly_pd["count"], color="steelblue")
axes[0,0].set_title("Piste A — Volume par heure")
axes[0,0].set_xlabel("Heure")
axes[0,0].set_ylabel("Nb transactions")

# --- Piste B : Top users burst ---
burst_pd = burst_users.toPandas()
axes[0,1].barh(burst_pd["user_id"], burst_pd["count"], color="orange")
axes[0,1].set_title("Piste B — Users burst (>5 txn / 2h)")
axes[0,1].set_xlabel("Nb transactions en burst")

# --- Piste C : Pays de livraison suspects ---
geo_pd = geo_suspect.groupBy("delivery_country").count() \
    .orderBy(col("count").desc()).toPandas()
axes[0,2].bar(geo_pd["delivery_country"], geo_pd["count"], color="crimson")
axes[0,2].set_title("Piste C — Livraisons pays à risque")
axes[0,2].set_xlabel("Pays")
axes[0,2].set_ylabel("Nb transactions")

# --- Piste D : Boxplot montants nouveaux vs anciens comptes ---
new_pd = df_new.select("amount_eur").toPandas()
old_pd = df_old.select("amount_eur").toPandas()
axes[1,0].boxplot([new_pd["amount_eur"], old_pd["amount_eur"]],
                  labels=["Nouveaux (<7j)", "Anciens (>=7j)"])
axes[1,0].set_title("Piste D — Distribution montants")
axes[1,0].set_ylabel("Montant (€)")

# --- Piste E : Distribution scores de risque ---
scores_pd = df_scored.groupBy("risk_score").count() \
    .orderBy("risk_score").toPandas()
axes[1,1].bar(scores_pd["risk_score"].astype(str), scores_pd["count"], color="purple")
axes[1,1].set_title("Piste E — Distribution scores de risque")
axes[1,1].set_xlabel("Score")
axes[1,1].set_ylabel("Nb transactions")

# --- Top 50 : tableau récapitulatif ---
top50_pd = top50.select("transaction_id", "user_id", "amount_eur", "risk_score") \
    .toPandas().head(10)
axes[1,2].axis('off')
table = axes[1,2].table(
    cellText=top50_pd.values,
    colLabels=top50_pd.columns,
    loc='center', cellLoc='center'
)
table.scale(1, 1.4)
axes[1,2].set_title("Piste E — Top 10 transactions suspectes")

plt.tight_layout()
plt.savefig("fraud_x42_visualisations.png", dpi=150, bbox_inches='tight')
plt.show()
print("Visualisations sauvegardées.")

ModuleNotFoundError: No module named 'matplotlib'